# 02 · Exploring the patterns
**Is street crime clustered, and does it follow street lighting?**

We map crime and lighting, compare crime across lighting classes, then measure
spatial clustering with Moran's I.

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

In [ ]:
study = gpd.read_file('../data/processed/study_area.gpkg')
study.shape

## Section 1 — Maps

Crime rates are very uneven: a few city-centre squares have rates many times
higher than the rest. `np.log1p()` (the log of 1 + the rate) spreads out the
colours so we can see the pattern everywhere.

In [ ]:
study['log_street_rate'] = np.log1p(study['street_rate'])
study['log_comparison_rate'] = np.log1p(study['comparison_rate'])

study.plot(column='log_street_rate', cmap='viridis', legend=True, figsize=(7, 7))
plt.title('Street crime per 1,000 residents (log scale)')
plt.axis('off')
plt.show()

Make the same map for the `slai` column. Where do high crime and high lighting coincide?

In [ ]:
study.plot(column='slai', cmap='viridis', legend=True, figsize=(7, 7))
plt.title('Street Light Availability Index')
plt.axis('off')
plt.show()

## Section 2 — Does crime rise or fall with lighting?

Each dot is a square, coloured by population.

In [ ]:
study.plot.scatter(x='slai', y='log_street_rate', c='population',
                   cmap='viridis', s=12, alpha=0.6, figsize=(7, 4))
plt.xlabel('Street Light Availability Index')
plt.ylabel('Street crime per 1,000 (log)')
plt.title('Street crime and street lighting')
plt.show()

Compare the median rates in the three lighting classes, for both crime groups.

In [ ]:
study.groupby('light_class')[['street_rate', 'comparison_rate']].median().loc[['Low', 'Medium', 'High']]

In [ ]:
study.boxplot(column='log_street_rate', by='light_class', figsize=(6, 4))
plt.suptitle('')
plt.title('Street crime by lighting class')
plt.ylabel('Street crime per 1,000 (log)')
plt.show()

The **Kruskal–Wallis test** checks whether the three classes differ more than
chance would suggest. It does not assume the rates are normally distributed.

In [ ]:
from scipy.stats import kruskal

for rate in ['street_rate', 'comparison_rate']:
    groups = [study.loc[study['light_class'] == c, rate] for c in ['Low', 'Medium', 'High']]
    print(rate, kruskal(*groups))

The **Spearman correlation** measures whether two variables rise together (using
ranks). If lighting matters for street crime but not for the comparison group,
the SLAI should correlate more strongly with street crime.

In [ ]:
study[['slai', 'street_rate', 'comparison_rate', 'population']].corr(method='spearman').round(2)

## Section 3 — Is crime spatially clustered? (Moran's I)

Moran's I compares each square with its neighbours. Values near +1 mean similar
values cluster together; near 0 means no spatial pattern.

`KNN.from_dataframe(study, k=8)` makes each square's 8 nearest squares its neighbours.

In [ ]:
from libpysal.weights import KNN
from esda.moran import Moran, Moran_Local

w = KNN.from_dataframe(study, k=8)
w.transform = 'r'

moran = Moran(study['log_street_rate'], w)
print("Moran's I:", round(moran.I, 3), ' p =', moran.p_sim)

**LISA** (local Moran's I) shows *where* the clusters are. High-High squares are
hotspots; Low-Low are coldspots. We keep only results with p < 0.05.

In [ ]:
lisa = Moran_Local(study['log_street_rate'], w)
labels = {1: 'High-High', 2: 'Low-High', 3: 'Low-Low', 4: 'High-Low'}
study['lisa'] = pd.Series(lisa.q).map(labels).values
study.loc[lisa.p_sim >= 0.05, 'lisa'] = 'Not significant'
study['lisa'].value_counts()

In [ ]:
colours = {'High-High': 'red', 'Low-Low': 'blue', 'High-Low': 'orange',
           'Low-High': 'lightblue', 'Not significant': 'lightgrey'}
study.plot(color=study['lisa'].map(colours), figsize=(7, 7))
plt.title('Street crime hotspots (LISA)')
plt.axis('off')
plt.show()